# Step 4A.9 — B3: масове кодування 379,036 CVE через SecBERT (embedding generation ONLY)
Жодного навчання classifier тут немає — лише encoder → embeddings → збереження + перевірка.
Розділення embedding generation і classifier training навмисне: якщо classifier покаже
нестабільність пізніше, можна аналізувати representation окремо від classifier.

**B3 spec (заморожено):**
- Encoder: `jackaduma/SecBERT`
- `max_length=256` (97.24% описів на аудит-вибірці без обрізання, Step 4A.8 — не всі 379,036 CVE перевірялись поштучно)
- Pooling: attention-mask weighted mean (лише реальні токени, не [PAD])
- L2 normalization
- `description_missing` НЕ додається (0/379,036 missing у frozen dataset)
- Дуже короткі description кодуються як звичайний текст, без спецобробки

Ядро: IDS Research.

In [1]:
import sys, subprocess, importlib.util
if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from transformers import AutoTokenizer, AutoModel

CTI_DIR = Path("/srv/ids_research/data/cti")
RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "jackaduma/SecBERT"
MAX_LENGTH = 256
BATCH_SIZE = 32
EXPECTED_N_ROWS = 379036
EMBEDDING_DIM = 768

GPU_ID = 3

if torch.cuda.is_available():
    device = torch.device(f"cuda:{GPU_ID}")
    print(f"GPU: {torch.cuda.get_device_name(GPU_ID)}")
else:
    device = torch.device("cpu")
    print("WARNING: CUDA unavailable, using CPU")

print(f"device={device}, MAX_LENGTH={MAX_LENGTH}, BATCH_SIZE={BATCH_SIZE}")

GPU: NVIDIA RTX A5500
device=cuda:3, MAX_LENGTH=256, BATCH_SIZE=32


In [2]:
# ---------- Завантаження frozen dataset + перевірка розміру ----------
ds = pd.read_parquet(CTI_DIR / "cve_forecast_dataset.parquet", columns=["cve_id", "description"])
assert len(ds) == EXPECTED_N_ROWS, f"actual={len(ds)}, expected={EXPECTED_N_ROWS} — dataset змінився!"
assert ds["cve_id"].is_unique, "cve_id не унікальні — щось не так із frozen dataset"
print(f"✅ {len(ds):,} рядків, cve_id унікальні")

n_empty = ds["description"].fillna("").str.strip().eq("").sum()
assert n_empty == 0, (
    f"Знайдено {n_empty} порожніх description — frozen dataset НЕ відповідає Step 4A.8! "
    f"description_missing НЕ реалізовано в цьому ноутбуці навмисне (рішення Step 4A.8: "
    f"0/379,036 missing) — якщо ця перевірка впала, спершу треба переглянути рішення, "
    f"а не просто продовжувати кодування."
)
print(f"✅ Порожніх description: {n_empty}")

✅ 379,036 рядків, cve_id унікальні
✅ Порожніх description: 0


In [3]:
# ---------- Завантаження SecBERT (СУВОРО локально, без звернення до інтернету) ----------
# local_files_only=True критично для відтворюваності: цей крок НЕ повинен непомітно
# підтягнути іншу версію файлів чи залежати від доступності HuggingFace у момент запуску.
# Модель уже закешована локально зі Step 4A.7 — якщо цей рядок впаде з помилкою "not found
# locally", значить кеш відсутній/пошкоджений, і це треба розібрати, а не мовчки качати заново.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
model = AutoModel.from_pretrained(MODEL_NAME, local_files_only=True).to(device)
model.eval()
print(f"Модель завантажена ЛОКАЛЬНО (local_files_only=True). Параметрів: "
      f"{sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: jackaduma/SecBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель завантажена ЛОКАЛЬНО (local_files_only=True). Параметрів: 83,450,880


In [4]:
# ---------- Masked mean pooling + L2 normalization (batch inference) ----------
def masked_mean_pooling(last_hidden_state, attention_mask):
    """Mean pooling ЛИШЕ по реальних токенах (attention_mask=1), [PAD] не впливає на середнє."""
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def encode_batch(texts):
    inputs = tokenizer(texts, return_tensors="pt", truncation=True, max_length=MAX_LENGTH,
                       padding=True).to(device)
    with torch.inference_mode():   # чистіше й дещо ощадливіше за пам'яттю, ніж no_grad(), для pure inference
        output = model(**inputs)
    emb = masked_mean_pooling(output.last_hidden_state, inputs["attention_mask"])
    emb = F.normalize(emb, p=2, dim=1)   # L2 normalization
    return emb.cpu().numpy()

# швидка перевірка на 3 прикладах перед повним прогоном
test_emb = encode_batch(ds["description"].iloc[:3].tolist())
print(f"Тестовий batch: shape={test_emb.shape}, норма кожного вектора (має бути ~1.0 після L2): "
      f"{np.linalg.norm(test_emb, axis=1)}")

Тестовий batch: shape=(3, 768), норма кожного вектора (має бути ~1.0 після L2): [1.         0.9999999  0.99999994]


In [5]:
# ---------- Повне кодування 379,036 CVE, батчами, зі збереженням прогресу ----------
import time

texts = ds["description"].tolist()
n = len(texts)
all_embeddings = np.zeros((n, EMBEDDING_DIM), dtype=np.float32)

t0 = time.time()
for start in range(0, n, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n)
    batch_emb = encode_batch(texts[start:end])
    all_embeddings[start:end] = batch_emb
    if (start // BATCH_SIZE) % 200 == 0:
        elapsed = time.time() - t0
        processed = end   # фактично оброблено, а не start+BATCH_SIZE (некоректно на останньому неповному батчі)
        rate = processed / max(elapsed, 1e-9)
        eta = (n - processed) / max(rate, 1e-9)
        print(f"  {end:,}/{n:,}  ({elapsed:.0f}с, ~{rate:.1f} CVE/с, ETA ~{eta/60:.1f} хв)")

total_time = time.time() - t0
print(f"\n✅ Кодування завершено за {total_time/60:.1f} хв ({n/total_time:.1f} CVE/с)")

  32/379,036  (0с, ~1830.0 CVE/с, ETA ~3.5 хв)
  6,432/379,036  (5с, ~1378.8 CVE/с, ETA ~4.5 хв)
  12,832/379,036  (11с, ~1183.6 CVE/с, ETA ~5.2 хв)
  19,232/379,036  (19с, ~1008.6 CVE/с, ETA ~5.9 хв)
  25,632/379,036  (27с, ~937.4 CVE/с, ETA ~6.3 хв)
  32,032/379,036  (35с, ~914.7 CVE/с, ETA ~6.3 хв)
  38,432/379,036  (43с, ~902.3 CVE/с, ETA ~6.3 хв)
  44,832/379,036  (50с, ~890.0 CVE/с, ETA ~6.3 хв)
  51,232/379,036  (57с, ~893.6 CVE/с, ETA ~6.1 хв)
  57,632/379,036  (65с, ~890.4 CVE/с, ETA ~6.0 хв)
  64,032/379,036  (72с, ~887.7 CVE/с, ETA ~5.9 хв)
  70,432/379,036  (79с, ~886.9 CVE/с, ETA ~5.8 хв)
  76,832/379,036  (88с, ~875.9 CVE/с, ETA ~5.8 хв)
  83,232/379,036  (96с, ~865.8 CVE/с, ETA ~5.7 хв)
  89,632/379,036  (102с, ~879.0 CVE/с, ETA ~5.5 хв)
  96,032/379,036  (110с, ~869.6 CVE/с, ETA ~5.4 хв)
  102,432/379,036  (119с, ~860.5 CVE/с, ETA ~5.4 хв)
  108,832/379,036  (127с, ~858.8 CVE/с, ETA ~5.2 хв)
  115,232/379,036  (135с, ~851.6 CVE/с, ETA ~5.2 хв)
  121,632/379,036  (143с, 

In [6]:
# ---------- Перевірки якості embeddings ----------
print("1. Кількість embeddings == 379,036?")
assert all_embeddings.shape[0] == EXPECTED_N_ROWS
print(f"   ✅ {all_embeddings.shape[0]:,}")

print("\n2. Dimension == 768?")
assert all_embeddings.shape[1] == EMBEDDING_DIM
print(f"   ✅ {all_embeddings.shape[1]}")

print("\n3. NaN?")
n_nan = np.isnan(all_embeddings).sum()
assert n_nan == 0, f"Знайдено {n_nan} NaN-значень!"
print(f"   ✅ 0")

print("\n4. Inf?")
n_inf = np.isinf(all_embeddings).sum()
assert n_inf == 0, f"Знайдено {n_inf} Inf-значень!"
print(f"   ✅ 0")

print("\n5. cve_id унікальні (повторна перевірка)?")
assert ds["cve_id"].is_unique
print(f"   ✅ так")

print("\n6. Порядок CVE відповідає dataset (embeddings[i] відповідає ds.iloc[i])?")
# Перевіряємо непрямо: перекодовуємо СПРАВДІ ВИПАДКОВІ (не фіксовані) індекси і звіряємо
# з масивом — детермінований seed для відтворюваності запуску, але вибірка реально випадкова.
rng = np.random.default_rng(42)
sample_idx = rng.choice(n, size=5, replace=False).tolist()
recheck_texts = [texts[i] for i in sample_idx]
recheck_emb = encode_batch(recheck_texts)
matches = [np.allclose(all_embeddings[i], recheck_emb[j], atol=1e-5) for j, i in enumerate(sample_idx)]
assert all(matches), f"Розбіжність порядку на індексах {sample_idx}: {matches}"
print(f"   ✅ перевірено на {len(sample_idx)} випадкових індексах: {sample_idx}")

print("\n7. Норма векторів ~1.0 (L2 normalization застосована)?")
norms = np.linalg.norm(all_embeddings, axis=1)
print(f"   min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}")
assert np.allclose(norms, 1.0, atol=1e-3), "L2-норма відхиляється від 1.0"
print("   ✅")

print("\n✅ Усі перевірки embeddings пройдено.")

1. Кількість embeddings == 379,036?
   ✅ 379,036

2. Dimension == 768?
   ✅ 768

3. NaN?
   ✅ 0

4. Inf?
   ✅ 0

5. cve_id унікальні (повторна перевірка)?
   ✅ так

6. Порядок CVE відповідає dataset (embeddings[i] відповідає ds.iloc[i])?
   ✅ перевірено на 5 випадкових індексах: [293354, 166350, 248104, 33828, 164128]

7. Норма векторів ~1.0 (L2 normalization застосована)?
   min=1.0000, max=1.0000, mean=1.0000
   ✅

✅ Усі перевірки embeddings пройдено.


In [7]:
# ---------- Збереження: embeddings (.npy) + cve_id mapping (.parquet) ----------
emb_path = RESULTS_DIR / "step4a9_b3_embeddings.npy"
mapping_path = RESULTS_DIR / "step4a9_b3_cve_id_mapping.parquet"

np.save(emb_path, all_embeddings)
ds[["cve_id"]].reset_index(drop=True).to_parquet(mapping_path, index=False)

print(f"Збережено:")
print(f"  {emb_path} — {all_embeddings.nbytes / 1e6:.1f} МБ, shape={all_embeddings.shape}")
print(f"  {mapping_path} — cve_id у ТОМУ Ж порядку, що й рядки в .npy (index i <-> cve_id[i])")
print(f"\nℹ Завантаження для подальшого використання (Step 4A.10, B3 classifier):")
print(f"  embeddings = np.load('{emb_path}')")
print(f"  cve_id_order = pd.read_parquet('{mapping_path}')['cve_id']")
print(f"  # embeddings[i] відповідає cve_id_order[i]")

Збережено:
  /home/ivan_r_opirskyy/notebooks/2 article/release/results/step4a9_b3_embeddings.npy — 1164.4 МБ, shape=(379036, 768)
  /home/ivan_r_opirskyy/notebooks/2 article/release/results/step4a9_b3_cve_id_mapping.parquet — cve_id у ТОМУ Ж порядку, що й рядки в .npy (index i <-> cve_id[i])

ℹ Завантаження для подальшого використання (Step 4A.10, B3 classifier):
  embeddings = np.load('/home/ivan_r_opirskyy/notebooks/2 article/release/results/step4a9_b3_embeddings.npy')
  cve_id_order = pd.read_parquet('/home/ivan_r_opirskyy/notebooks/2 article/release/results/step4a9_b3_cve_id_mapping.parquet')['cve_id']
  # embeddings[i] відповідає cve_id_order[i]


In [8]:
emb = np.load(emb_path, mmap_mode="r")
mapping = pd.read_parquet(mapping_path)

print(emb.shape)
print(len(mapping))
print(mapping["cve_id"].is_unique)

assert emb.shape == (379036, 768)
assert len(mapping) == 379036
assert mapping["cve_id"].is_unique
assert np.isnan(emb).sum() == 0
assert np.isinf(emb).sum() == 0

(379036, 768)
379036
True


In [10]:
assert mapping["cve_id"].tolist() == ds["cve_id"].tolist()